
# ServiFlow – Génération des données (v2 améliorée)
**Cas pratique BI / Data : segmentation clients + churn (abonnement B2B SaaS)**

Ce notebook génère **4 tables** en modèle étoile :
- `dim_client.csv` : 1 ligne = 1 client (inclut `date_fin_abonnement` si résiliation)
- `dim_offre.csv` : 1 ligne = 1 offre (prix de base + paramètres tarifaires)
- `dim_date.csv` : 1 ligne = 1 mois
- `fact_abonnement_mensuel.csv` : 1 ligne = client × mois

## Objectifs de réalisme (pédagogie)
- Distributions **non linéaires** et **corrélations crédibles** (MRR ↔ licences ↔ usage, support ↔ satisfaction, etc.)
- Quelques **valeurs manquantes** et **outliers** (pour tester un preprocessing robuste)
- Signaux churn plausibles (sous-usage, baisse satisfaction, incidents, retards paiement), avec **non-linéarités** et **interactions**
- Segments “latents” partiellement **chevauchants** : hésitation naturelle entre 3/4/5 clusters

> La variable cible n’est **pas fournie** dans la table de faits : le churn est **dérivé** via `dim_client.date_fin_abonnement`.


In [2]:
%pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 18.4 MB/s eta 0:00:00


In [1]:

import numpy as np
import pandas as pd
import re
from faker import Faker

fake = Faker("fr_FR")


ModuleNotFoundError: No module named 'faker'


## 1) Paramètres
Vous pouvez ajuster le volume et le niveau de “bruit” sans casser le réalisme global.


In [ ]:

# ----------------------------
# Paramètres de génération
# ----------------------------
SEED_GENERATION = 42
rng = np.random.default_rng(SEED_GENERATION)

NB_CLIENTS = 1800            # 1 à 2k : bon compromis pédagogie / performance
NB_MOIS_HISTO = 18           # 18 mois : permet churn + cohortes + tendances
DATE_DEBUT_HISTO = pd.Timestamp("2024-01-01")
DATES_MENSUELLES = pd.date_range(DATE_DEBUT_HISTO, periods=NB_MOIS_HISTO, freq="MS")

# Churn : taux mensuel moyen cible (hazard) sur mois éligibles (hors lock-in)
CIBLE_CHURN_MENSUEL = 0.010  # ≈ 1% / mois

# Clients atypiques (mélange) : rendent les clusters moins “propres”
TAUX_CLIENTS_ATYPIQUES = 0.10

# Qualité (valeurs manquantes / outliers)
TAUX_NA_USAGE = 0.03
TAUX_NA_SAT = 0.03
TAUX_NA_MRR = 0.02

NB_OUTLIERS_MRR = 15
NB_OUTLIERS_USAGE = 15
NB_OUTLIERS_TICKETS = 10

MOIS_FR = {
    1: "janvier", 2: "février", 3: "mars", 4: "avril", 5: "mai", 6: "juin",
    7: "juillet", 8: "août", 9: "septembre", 10: "octobre", 11: "novembre", 12: "décembre"
}



## 2) Fonctions utilitaires


In [ ]:

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def mois_entre(d1: pd.Timestamp, d2: pd.Timestamp) -> int:
    """Différence en mois entre deux timestamps (d2 - d1)."""
    return (d2.year - d1.year) * 12 + (d2.month - d1.month)

def clip(x, a, b):
    return max(a, min(b, x))



## 3) Génération des dimensions (date, offre, client)


In [ ]:

def generer_dim_date():
    dim_date = pd.DataFrame({
        "id_date": [d.year * 100 + d.month for d in DATES_MENSUELLES],
        "date_mois": DATES_MENSUELLES,
        "annee": [d.year for d in DATES_MENSUELLES],
        "mois": [d.month for d in DATES_MENSUELLES],
        "nom_mois": [MOIS_FR[d.month] for d in DATES_MENSUELLES],
    })
    dim_date["trimestre"] = ((dim_date["mois"] - 1) // 3) + 1
    return dim_date

def generer_dim_offre():
    # Prix : modèle simple base + prix par licence (seat-based)
    # -> rend le MRR cohérent avec nb_licences
    return pd.DataFrame([
        {"id_offre": 1, "nom_offre": "Essentiel",  "niveau_sla": "Standard",
         "prix_mensuel_base_eur": 220, "prix_par_licence_eur": 25,  "remise_volume_max_pct": 0.10},
        {"id_offre": 2, "nom_offre": "Pro",        "niveau_sla": "Étendu",
         "prix_mensuel_base_eur": 520, "prix_par_licence_eur": 55,  "remise_volume_max_pct": 0.12},
        {"id_offre": 3, "nom_offre": "Entreprise", "niveau_sla": "24/7",
         "prix_mensuel_base_eur": 1500,"prix_par_licence_eur": 85,  "remise_volume_max_pct": 0.15},
        {"id_offre": 4, "nom_offre": "Sur-mesure", "niveau_sla": "24/7",
         "prix_mensuel_base_eur": 2600,"prix_par_licence_eur": 120, "remise_volume_max_pct": 0.18},
    ])





def generer_dim_client(nb_clients: int, seed: int = 42):
    # Génère des clients avec un nom de société cohérent avec le pays, le secteur et le segment.
    # Réalisme ajouté :
    # - Faker localisé par pays (France/Allemagne/Espagne/Italie/Belgique)
    # - Préfixes localisés pour "grands comptes" (Gruppe/Grupo/Gruppo/Groupe/Group)
    # - Tokens sectoriels : FR/BE (mix FR/EN), autres pays (EN neutre)
    # - Formes juridiques par pays, sans doublons (évite 'GmbH AG', etc.)

    rng_local = np.random.default_rng(seed)

    segments = [
        "A_petits_opportunistes",
        "B_PME_en_croissance",
        "C_grands_comptes_sous_exploites",
        "D_grands_comptes_strategiques"
    ]
    proba_segments = [0.35, 0.30, 0.20, 0.15]

    secteurs = ["Informatique", "Commerce de détail", "Santé", "Industrie", "Secteur public", "Services professionnels"]
    pays = ["France", "Allemagne", "Espagne", "Italie", "Belgique"]

    def _mk_faker(locale: str):
        try:
            return Faker(locale)
        except Exception:
            return Faker("fr_FR")

    fakers = {
        "France": _mk_faker("fr_FR"),
        "Allemagne": _mk_faker("de_DE"),
        "Espagne": _mk_faker("es_ES"),
        "Italie": _mk_faker("it_IT"),
        "Belgique": _mk_faker("fr_BE"),
    }

    # Formes juridiques par pays (simplifiées)
    formes = {
        "France": ["SAS", "SARL", "SA"],
        "Belgique": ["SRL", "SA", "NV"],
        "Allemagne": ["GmbH", "AG"],
        "Espagne": [r"S\.L\.", r"S\.A\.", r"SL", r"SA"],
            "Italie": [r"S\.R\.L\.", r"S\.P\.A\.", r"SRL", r"SPA"],
    }

    tokens_secteur_fr = {
        "Informatique": ["Digital", "Data", "Cloud", "Technologies", "Solutions", "Systèmes"],
        "Commerce de détail": ["Retail", "Distribution", "Marchés", "Commerces", "Magasins", "Supply"],
        "Santé": ["Santé", "Médical", "Care", "Pharma", "Clinique", "Bio"],
        "Industrie": ["Industries", "Manufacturing", "Mécanique", "Énergie", "Production", "Engineering"],
        "Secteur public": ["Collectivités", "Territoires", "Services", "Établissements", "Régie", "Agence"],
        "Services professionnels": ["Conseil", "Audit", "Partners", "Expertise", "Services", "Associés"],
    }
    tokens_secteur_int = {
        "Informatique": ["Digital", "Data", "Cloud", "Technologies", "Solutions", "Systems"],
        "Commerce de détail": ["Retail", "Distribution", "Commerce", "Supply", "Stores", "Markets"],
        "Santé": ["Health", "Medical", "Care", "Pharma", "Clinic", "Bio"],
        "Industrie": ["Industry", "Manufacturing", "Engineering", "Energy", "Production", "Industrial"],
        "Secteur public": ["Public Services", "Agency", "Territory", "Administration", "Civic", "Public"],
        "Services professionnels": ["Consulting", "Advisory", "Audit", "Partners", "Services", "Expertise"],
    }

    prefixes_segment_par_pays = {
        "France": {
            "A_petits_opportunistes": ["", "", ""],
            "B_PME_en_croissance": ["", "Société", ""],
            "C_grands_comptes_sous_exploites": ["Groupe", "Holding", "Groupe"],
            "D_grands_comptes_strategiques": ["Groupe", "International", "Groupe"],
        },
        "Belgique": {
            "A_petits_opportunistes": ["", "", ""],
            "B_PME_en_croissance": ["", "Société", ""],
            "C_grands_comptes_sous_exploites": ["Groupe", "Holding", "Group"],
            "D_grands_comptes_strategiques": ["Groupe", "International", "Group"],
        },
        "Allemagne": {
            "A_petits_opportunistes": ["", "", ""],
            "B_PME_en_croissance": ["", "", ""],
            "C_grands_comptes_sous_exploites": ["Gruppe", "Holding", "Group"],
            "D_grands_comptes_strategiques": ["Gruppe", "International", "Group"],
        },
        "Espagne": {
            "A_petits_opportunistes": ["", "", ""],
            "B_PME_en_croissance": ["", "", ""],
            "C_grands_comptes_sous_exploites": ["Grupo", "Holding", "Group"],
            "D_grands_comptes_strategiques": ["Grupo", "Internacional", "Group"],
        },
        "Italie": {
            "A_petits_opportunistes": ["", "", ""],
            "B_PME_en_croissance": ["", "", ""],
            "C_grands_comptes_sous_exploites": ["Gruppo", "Holding", "Group"],
            "D_grands_comptes_strategiques": ["Gruppo", "Internazionale", "Group"],
        },
    }

    date_ref = pd.Timestamp("2024-12-01")
    ids_clients = [fake.uuid4() for _ in range(nb_clients)]
    seg_attribue = rng_local.choice(segments, size=nb_clients, p=proba_segments)

    def tirer_anciennete():
        return int(clip(rng_local.exponential(scale=26) + 1, 1, 84))

    def nettoyer_nom(nom: str) -> str:
        return re.sub(r"\s+", " ", str(nom)).strip()

    def contient_forme_juridique(nom: str, pays_client: str) -> bool:
        nom_u = nom.upper()
        patterns = {
            "France": [r"\bSAS\b", r"\bSARL\b", r"\bSA\b", r"\bS\.A\.\b"],
            "Belgique": [r"\bSRL\b", r"\bSA\b", r"\bNV\b"],
            "Allemagne": [r"\bGMBH\b", r"\bMBH\b", r"\bGGMBH\b", r"\bAG\b", r"\bKG\b", r"\bOHG\b", r"\bGBR\b", r"\bE\.V\.\b", r"\bEV\b", r"\bSTIFTUNG\b", r"\bKGAA\b"],
            "Espagne": [r"S\.L\.", r"S\.A\.", r"SL", r"SA"],
            "Italie": [r"S\.R\.L\.", r"S\.P\.A\.", r"SRL", r"SPA"],
        }
        for p in patterns.get(pays_client, []):
            if re.search(p, nom_u):
                return True
        return False

    def choisir_forme(pays_client: str, seg: str) -> str:
        pool = formes.get(pays_client, ["SA"])
        if seg in ["C_grands_comptes_sous_exploites", "D_grands_comptes_strategiques"]:
            pref = [x for x in pool if x in ["SA", "AG", "S.A.", "S.p.A."]]
            if pref:
                return str(rng_local.choice(pref))
        return str(rng_local.choice(pool))

    def generer_nom_societe(pays_client: str, secteur: str, seg: str) -> str:
        fk = fakers.get(pays_client, fakers["France"])
        core = nettoyer_nom(fk.company())

        token_pool = tokens_secteur_fr[secteur] if pays_client in ["France", "Belgique"] else tokens_secteur_int[secteur]
        tok = str(rng_local.choice(token_pool))

        pref_pool = prefixes_segment_par_pays.get(pays_client, prefixes_segment_par_pays["France"])[seg]
        pref = str(rng_local.choice(pref_pool))

        nom = f"{pref} {core}" if pref else core

        if rng_local.random() < 0.30:
            nom = f"{nom} {tok}"
        else:
            nom = f"{tok} {nom}" if rng_local.random() < 0.50 else f"{nom} {tok}"

        if seg in ["C_grands_comptes_sous_exploites", "D_grands_comptes_strategiques"] and rng_local.random() < 0.35:
            marque = nettoyer_nom(fk.company().split(" ")[0])
            pref0 = pref_pool[0] if len(pref_pool) > 0 else "Group"
            nom = f"{pref0} {marque} {tok}"

        nom = nettoyer_nom(nom)

        if not contient_forme_juridique(nom, pays_client):
            forme = choisir_forme(pays_client, seg)
            if rng_local.random() < (0.65 if seg in ["A_petits_opportunistes", "B_PME_en_croissance"] else 0.35):
                nom = f"{nom} {forme}"

        return nettoyer_nom(nom)

    rows = []
    for cid, seg in zip(ids_clients, seg_attribue):
        secteur = str(rng_local.choice(secteurs))
        pays_client = str(rng_local.choice(pays, p=[0.65, 0.10, 0.10, 0.10, 0.05]))

        if seg == "A_petits_opportunistes":
            nb_employes = int(rng_local.integers(5, 90))
        elif seg == "B_PME_en_croissance":
            nb_employes = int(rng_local.integers(30, 450))
        elif seg == "C_grands_comptes_sous_exploites":
            nb_employes = int(rng_local.integers(250, 7000))
        else:
            nb_employes = int(rng_local.integers(400, 9000))

        anciennete_mois = tirer_anciennete()
        date_debut = (date_ref - pd.DateOffset(months=anciennete_mois)).normalize()

        rows.append({
            "id_client": cid,
            "nom_client": generer_nom_societe(pays_client, secteur, seg),
            "secteur_activite": secteur,
            "pays": pays_client,
            "nb_employes": nb_employes,
            "date_debut_abonnement": date_debut.date(),
            "segment_reference": seg,
        })

    dim_client = pd.DataFrame(rows)

    if dim_client["nom_client"].duplicated().any():
        dup = dim_client["nom_client"].duplicated(keep=False)
        dim_client.loc[dup, "nom_client"] = (
            dim_client.loc[dup, "nom_client"]
            + " "
            + dim_client.loc[dup].groupby("nom_client").cumcount().astype(str)
        )

    return dim_client, ids_clients, seg_attribue







## 4) Génération de la table de faits + date de fin d’abonnement (churn)
On simule :
- **licences** liées à la taille client (nb_employes) + adoption
- **usage** lié aux licences + engagement + saisonnalité + tendance
- **tickets / escalades** liés à usage + complexité + incidents
- **satisfaction** qui baisse avec tickets/escalades/retards (relation attendue)
- **retard paiement** plus probable pour petits clients/forte sensibilité prix

Churn :
- événement mensuel (hazard) avec lock-in selon l’offre
- non-linéarités : seuils de satisfaction / sous-usage ; interactions (incidents + faible satisfaction)
- état de “dégradation” (drift) pour générer des signaux pré-churn.


In [ ]:

def calibrer_beta0(cible_churn_mensuel: float, seed: int = 123) -> float:
    """Calibrage grossier de beta0 pour obtenir une moyenne de proba proche de la cible."""
    rng_local = np.random.default_rng(seed)
    n = 160_000

    segs = rng_local.choice(["A", "B", "C", "D"], size=n, p=[0.35, 0.30, 0.20, 0.15])
    biais_seg = np.select([segs=="A", segs=="B", segs=="C", segs=="D"], [0.60, 0.20, -0.05, -0.55])

    # covariables synthétiques
    usage_par_lic = np.clip(rng_local.normal(2.4, 0.9, size=n), 0.2, 6.0)
    satisfaction = np.clip(rng_local.normal(6.7, 1.5, size=n), 0, 10)
    tickets = rng_local.poisson(3.0, size=n)
    escalades = rng_local.binomial(np.minimum(tickets, 20), p=0.07, size=n)
    retard = (rng_local.random(n) < 0.06).astype(int)
    anciennete = rng_local.integers(1, 60, size=n)
    effet_debut = (anciennete <= 6).astype(int)

    # non-linéarités
    sous_usage = (usage_par_lic < 1.2).astype(int)
    sat_basse = (satisfaction < 4.5).astype(int)
    tickets_haut = (tickets >= 8).astype(int)
    inter_incident = (sat_basse & tickets_haut).astype(int)
    escalades_haut = (escalades >= 2).astype(int)

    # clients “protégés” (contrats / relation CSM) -> pas de churn ce mois
    protege = (rng_local.random(n) < 0.10).astype(int)

    z_sans_beta0 = (
        biais_seg
        + 0.45 * effet_debut
        + 0.90 * sous_usage
        + 1.20 * sat_basse
        + 0.55 * tickets_haut
        + 0.80 * inter_incident
        + 0.35 * escalades_haut
        + 0.50 * retard
        + rng_local.normal(0, 0.35, size=n)
    )

    def taux(beta0):
        p = sigmoid(beta0 + z_sans_beta0)
        p = np.where(protege == 1, 0.0, p)
        return float(p.mean())

    lo, hi = -8.0, -0.5
    for _ in range(30):
        mid = (lo + hi) / 2
        if taux(mid) > cible_churn_mensuel:
            hi = mid
        else:
            lo = mid
    return (lo + hi) / 2


def generer_fact_et_dates_fin(dim_client: pd.DataFrame, ids_clients, segs, dim_offre: pd.DataFrame, seed: int, beta0: float):
    rng_local = np.random.default_rng(seed)

    # Pré-calculs (performance) : accès O(1) par id_client
    _tmp = dim_client.set_index("id_client")
    debut_map = pd.to_datetime(_tmp["date_debut_abonnement"], errors="coerce")
    nb_emp_map = _tmp["nb_employes"].to_dict()

    # Biais segment (churn & patterns)
    biais_seg = {
        "A_petits_opportunistes": 0.60,
        "B_PME_en_croissance": 0.20,
        "C_grands_comptes_sous_exploites": -0.05,
        "D_grands_comptes_strategiques": -0.55,
    }

    # Adoption (licences) et intensité d’usage par licence (heures/licence/mois)
    # -> segments partiellement chevauchants (bruit + atypiques)
    params_adoption = {
        "A_petits_opportunistes": dict(adoption=0.18, usage_mu=2.1, usage_sd=0.8, complex_mu=0.45),
        "B_PME_en_croissance": dict(adoption=0.28, usage_mu=2.6, usage_sd=0.9, complex_mu=0.50),
        "C_grands_comptes_sous_exploites": dict(adoption=0.10, usage_mu=1.7, usage_sd=0.8, complex_mu=0.65),  # sous-usage relatif
        "D_grands_comptes_strategiques": dict(adoption=0.35, usage_mu=2.8, usage_sd=1.0, complex_mu=0.70),
    }

    # Distribution offres initiales (mix commercial)
    proba_offres = {
        "A_petits_opportunistes":          ([1, 2, 3],       [0.72, 0.23, 0.05]),
        "B_PME_en_croissance":             ([1, 2, 3, 4],    [0.12, 0.66, 0.20, 0.02]),
        "C_grands_comptes_sous_exploites": ([2, 3, 4],       [0.12, 0.63, 0.25]),
        "D_grands_comptes_strategiques":   ([3, 4],          [0.28, 0.72]),
    }

    # Certains clients changent d'offre (upgrade/downgrade) : réalisme & non-linéarité
    TAUX_CHANGEMENT_OFFRE = 0.12

    # Atypiques (bruit) : mélangent les centres
    mask_atypique = (rng_local.random(len(ids_clients)) < TAUX_CLIENTS_ATYPIQUES)

    date_fin_abonnement = {cid: pd.NaT for cid in ids_clients}
    fact_rows = []

    # Pré-calc offres
    offre_map = dim_offre.set_index("id_offre").to_dict(orient="index")

    for i, (cid, seg) in enumerate(zip(ids_clients, segs)):
        pseg = params_adoption[seg]

        # Sensibilité prix (drive retards et churn)
        if seg == "A_petits_opportunistes":
            sens_prix = float(np.clip(rng_local.beta(3, 4), 0, 1))
        elif seg == "B_PME_en_croissance":
            sens_prix = float(np.clip(rng_local.beta(2, 4), 0, 1))
        else:
            sens_prix = float(np.clip(rng_local.beta(2, 6), 0, 1))

        complexite = float(np.clip(rng_local.normal(pseg["complex_mu"], 0.15), 0.1, 0.95))
        engagement = float(np.clip(rng_local.beta(3, 3), 0.05, 0.98))

        # Offre initiale
        offres, p = proba_offres[seg]
        id_offre = int(rng_local.choice(offres, p=p))

        # Date de début client (peut être antérieure à la fenêtre observée)
        debut = pd.Timestamp(debut_map.loc[cid])
        debut_mois = pd.Timestamp(debut.year, debut.month, 1)
        debut_mois = max(debut_mois, DATE_DEBUT_HISTO)

        # Licences : fonction de nb_employes × adoption (avec bruit)
        nb_emp = int(nb_emp_map[cid])
        base_lic = max(1, int(round(nb_emp * pseg["adoption"] * rng_local.lognormal(0, 0.25))))

        # Atypiques : adoption plus erratique
        if mask_atypique[i]:
            base_lic = max(1, int(round(base_lic * rng_local.uniform(0.5, 1.8))))
            engagement = float(np.clip(engagement + rng_local.normal(0, 0.20), 0.05, 0.98))
            complexite = float(np.clip(complexite + rng_local.normal(0, 0.20), 0.1, 0.95))

        # Plan change : moment + direction
        changement = (rng_local.random() < TAUX_CHANGEMENT_OFFRE)
        mois_changement = None
        if changement:
            # pas trop tôt : après quelques mois observés
            mois_changement = int(rng_local.integers(5, max(6, NB_MOIS_HISTO - 2)))

        # Etat “dégradation” : augmente le risque (simulateur de drift)
        degrade = False

        resilie = False
        for idx_mois, d in enumerate(DATES_MENSUELLES):
            if d < debut_mois:
                continue
            if resilie:
                break

            # Changement d’offre
            if mois_changement is not None and idx_mois == mois_changement:
                # upgrade si engagement et croissance ; downgrade si sens_prix + faible engagement
                if (seg in ["B_PME_en_croissance", "D_grands_comptes_strategiques"]) and (engagement > 0.55) and (id_offre < 4):
                    id_offre = id_offre + 1
                elif (sens_prix > 0.65) and (engagement < 0.40) and (id_offre > 1):
                    id_offre = id_offre - 1

            # Ancienneté (mois) depuis début de la relation (au niveau mois)
            anciennete_mois = mois_entre(debut_mois, d) + 1

            # Lock-in selon offre (engagement contractuel)
            if id_offre in (1, 2) and anciennete_mois <= 2:
                churn_possible = False
            elif id_offre in (3, 4) and anciennete_mois <= 5:
                churn_possible = False
            else:
                churn_possible = True

            # Saisonnalité simple (cycle annuel)
            saison = 0.20 * np.sin(2 * np.pi * (d.month / 12))

            # Licences : dérive (croissance / réduction) + bruit léger
            if seg == "B_PME_en_croissance":
                drift = 1 + 0.02 * min(anciennete_mois, 24) / 12
            elif seg == "A_petits_opportunistes":
                drift = 1 - 0.01 * min(anciennete_mois, 18) / 12
            else:
                drift = 1.0

            licences = int(max(1, round(base_lic * drift + rng_local.normal(0, max(2, base_lic * 0.05)))))

            # Usage par licence (heures/licence/mois)
            usage_par_lic = float(np.clip(rng_local.normal(pseg["usage_mu"] + 0.8 * (engagement - 0.5) - 0.4 * (complexite - 0.5), pseg["usage_sd"]), 0.2, 6.0))
            usage_par_lic *= (1 + saison)

            # Incidents (choc) : augmente tickets/escalades et baisse satisfaction
            incident = (rng_local.random() < (0.02 + 0.06 * complexite))

            # Drift dégradation : activation rare ; persistance partielle
            if not degrade and churn_possible and (rng_local.random() < (0.01 + 0.04 * sens_prix + 0.02 * (complexite > 0.7))):
                degrade = True
            if degrade and (rng_local.random() < 0.15):  # résolution possible
                degrade = False

            # Usage mensuel total
            usage = licences * usage_par_lic + rng_local.normal(0, 6)
            if incident:
                usage *= rng_local.uniform(0.85, 0.98)  # incident : légère baisse d’usage
            if degrade:
                usage *= rng_local.uniform(0.55, 0.80)  # dégradation : forte baisse

            usage = float(max(0, usage))

            # Tickets support : Poisson avec lambda dépendant de l’usage, de la complexité et des incidents
            lam_tickets = 0.03 * usage + 0.015 * licences + 1.2 * complexite
            if incident:
                lam_tickets += rng_local.uniform(3, 8)
            if degrade:
                lam_tickets += rng_local.uniform(1, 4)

            tickets = int(rng_local.poisson(lam=max(0.2, lam_tickets)))

            # Escalades : corrélées aux tickets + incidents
            p_escalade = 0.05 + 0.10 * (incident) + 0.06 * complexite
            escalades = int(rng_local.binomial(n=min(tickets, 25), p=clip(p_escalade, 0.02, 0.40)))

            # MRR : base + prix par licence, discount volume, bruit multiplicatif
            o = offre_map[id_offre]
            base = float(o["prix_mensuel_base_eur"])
            p_lic = float(o["prix_par_licence_eur"])
            mrr_theorique = base + p_lic * licences

            # Remises volume (non-linéaires)
            remise = 0.0
            if licences >= 120:
                remise = min(o["remise_volume_max_pct"], 0.18)
            elif licences >= 60:
                remise = min(o["remise_volume_max_pct"], 0.12)
            elif licences >= 30:
                remise = min(o["remise_volume_max_pct"], 0.08)

            # Remise fidélité légère (tenure)
            if anciennete_mois >= 24:
                remise += 0.03

            remise = float(clip(remise, 0.0, 0.22))

            bruit_mrr = float(rng_local.lognormal(mean=0.0, sigma=0.08))
            if mask_atypique[i] and (rng_local.random() < 0.25):
                bruit_mrr *= float(rng_local.lognormal(mean=0.0, sigma=0.25))

            mrr = float(max(150, mrr_theorique * (1 - remise) * bruit_mrr))

            # Retard paiement : plus probable si sens_prix élevée, petit client, et mrr “lourd” vs taille
            ratio_mrr_par_emp = mrr / max(10, nb_emp)
            z_retard = -2.7 + 2.0 * sens_prix + 0.8 * (seg == "A_petits_opportunistes") + 0.4 * (ratio_mrr_par_emp > 6)
            p_retard = float(sigmoid(z_retard))
            indicateur_retard = int(rng_local.random() < p_retard)

            # Satisfaction : baisse avec tickets/escalades/retard, baisse si incident/dégradation
            base_sat = 6.2
            if seg == "B_PME_en_croissance":
                base_sat = 6.8
            elif seg == "C_grands_comptes_sous_exploites":
                base_sat = 6.5
            elif seg == "D_grands_comptes_strategiques":
                base_sat = 7.2

            sat = (
                base_sat
                + 0.25 * (engagement - 0.5) * 4
                + 0.06 * (usage_par_lic - 2.2) * 5
                - 0.12 * tickets
                - 0.45 * escalades
                - 0.65 * indicateur_retard
                + rng_local.normal(0, 0.9)
            )
            if incident:
                sat -= rng_local.uniform(0.7, 1.6)
            if degrade:
                sat -= rng_local.uniform(0.8, 1.8)

            satisfaction = float(np.clip(sat, 0, 10))

            # Sous-usage relatif : ratio usage/(licences*2.4) ~ 1 normal ; < 0.6 sous-usage
            ratio_usage = usage / max(1.0, licences * 2.4)

            # Churn logit : non-linéaire + interactions + bruit
            sous_usage = int(ratio_usage < 0.60)
            sat_basse = int(satisfaction < 4.5)
            tickets_haut = int(tickets >= 8)
            inter = int((sous_usage == 1) and (sat_basse == 1))
            inter2 = int((sat_basse == 1) and (tickets_haut == 1))
            escalades_haut = int(escalades >= 2)
            effet_debut = int(anciennete_mois <= 6)

            protege = int((mrr > 9000) and (satisfaction > 7.5) and (seg in ["C_grands_comptes_sous_exploites", "D_grands_comptes_strategiques"]) and (rng_local.random() < 0.20))

            z = (
                beta0
                + biais_seg[seg]
                + 0.45 * effet_debut
                + 0.90 * sous_usage
                + 1.20 * sat_basse
                + 0.55 * tickets_haut
                + 0.80 * inter2
                + 0.55 * inter
                + 0.35 * escalades_haut
                + 0.50 * indicateur_retard
                + (0.35 if incident else 0.0)
                + (0.55 if degrade else 0.0)
                - (0.60 if protege else 0.0)
                + rng_local.normal(0, 0.35)
            )

            proba_churn = float(sigmoid(z))
            churn_event = 0
            if churn_possible and (protege == 0):
                churn_event = int(rng_local.random() < proba_churn)

            if churn_event == 1:
                resilie = True
                date_fin = (d + pd.offsets.MonthEnd(0)).normalize()
                date_fin_abonnement[cid] = date_fin

            fact_rows.append({
                "id_client": cid,
                "id_offre": int(id_offre),
                "id_date": int(d.year * 100 + d.month),
                "mrr_eur": float(mrr),
                "nb_licences": int(licences),
                "heures_utilisation": float(usage),
                "nb_tickets_support": int(tickets),
                "nb_escalades": int(escalades),
                "score_satisfaction": float(satisfaction),
                "indicateur_retard_paiement": int(indicateur_retard),
            })

    fact = pd.DataFrame(fact_rows)
    return fact, date_fin_abonnement


def injecter_bruit_qualite(fact: pd.DataFrame, seed: int) -> pd.DataFrame:
    rng_local = np.random.default_rng(seed)
    fact = fact.copy()

    def injecter_na(col, pct):
        idx = rng_local.choice(fact.index, size=int(pct * len(fact)), replace=False)
        fact.loc[idx, col] = np.nan

    injecter_na("heures_utilisation", TAUX_NA_USAGE)
    injecter_na("score_satisfaction", TAUX_NA_SAT)
    injecter_na("mrr_eur", TAUX_NA_MRR)

    # Outliers (quelques lignes très atypiques)
    idx_mrr = rng_local.choice(fact.index, size=NB_OUTLIERS_MRR, replace=False)
    fact.loc[idx_mrr, "mrr_eur"] *= rng_local.uniform(6, 10, size=NB_OUTLIERS_MRR)

    idx_usage = rng_local.choice(fact.index, size=NB_OUTLIERS_USAGE, replace=False)
    fact.loc[idx_usage, "heures_utilisation"] *= rng_local.uniform(4, 7, size=NB_OUTLIERS_USAGE)

    idx_tickets = rng_local.choice(fact.index, size=NB_OUTLIERS_TICKETS, replace=False)
    fact.loc[idx_tickets, "nb_tickets_support"] += rng_local.integers(20, 40, size=NB_OUTLIERS_TICKETS)

    return fact



## 5) Exécution de la génération + export CSV


In [ ]:

# ----------------------------
# Génération complète + calibrage du churn
# ----------------------------
dim_date = generer_dim_date()
dim_offre = generer_dim_offre()
dim_client, ids_clients, segs = generer_dim_client(NB_CLIENTS, seed=SEED_GENERATION)

# 1) calibrage initial (approx)
beta0 = calibrer_beta0(CIBLE_CHURN_MENSUEL, seed=123)

# 2) Ajustement itératif sur le hazard réellement observé (avec lock-in + drift)
#    Méthode : on simule, on calcule le hazard observé sur mois éligibles, puis on ajuste beta0 via un décalage logit.
def calculer_hazard_observe(fact_tmp: pd.DataFrame, dim_client_tmp: pd.DataFrame, dim_date_tmp: pd.DataFrame) -> float:
    dd = dim_date_tmp.copy()
    dd["date_mois"] = pd.to_datetime(dd["date_mois"], errors="coerce")
    dc = dim_client_tmp.copy()
    dc["date_debut_abonnement"] = pd.to_datetime(dc["date_debut_abonnement"], errors="coerce")
    dc["date_fin_abonnement"] = pd.to_datetime(dc["date_fin_abonnement"], errors="coerce")

    fa = fact_tmp.merge(dd[["id_date", "date_mois"]], on="id_date", how="left")
    fa = fa.merge(dc[["id_client", "date_debut_abonnement", "date_fin_abonnement"]], on="id_client", how="left")

    fa["debut_mois"] = fa["date_debut_abonnement"].dt.to_period("M")
    fa["periode"] = fa["date_mois"].dt.to_period("M")
    fa["anciennete_mois"] = (fa["periode"].dt.year - fa["debut_mois"].dt.year) * 12 + (fa["periode"].dt.month - fa["debut_mois"].dt.month) + 1

    fa["churn_possible"] = True
    fa.loc[fa["id_offre"].isin([1, 2]) & (fa["anciennete_mois"] <= 2), "churn_possible"] = False
    fa.loc[fa["id_offre"].isin([3, 4]) & (fa["anciennete_mois"] <= 5), "churn_possible"] = False

    # churn_event : mois correspondant à date_fin_abonnement (si résilié)
    fa["churn_event"] = (
        fa["date_fin_abonnement"].notna()
        & (fa["date_fin_abonnement"].dt.to_period("M") == fa["periode"])
    ).astype(int)

    eligible = fa["churn_possible"].sum()
    events = fa["churn_event"].sum()
    return float(events / max(1, eligible))

def logit(p):
    p = float(np.clip(p, 1e-6, 1 - 1e-6))
    return np.log(p / (1 - p))

print(f"[INFO] beta0 initial ≈ {beta0:.3f} (cible churn mensuel {CIBLE_CHURN_MENSUEL*100:.2f}%)")

for it in range(2):  # 2 itérations suffisent en pratique
    fact_tmp, map_date_fin = generer_fact_et_dates_fin(
        dim_client=dim_client,
        ids_clients=ids_clients,
        segs=segs,
        dim_offre=dim_offre,
        seed=SEED_GENERATION + it,
        beta0=beta0
    )
    dc_tmp = dim_client.copy()
    dc_tmp["date_fin_abonnement"] = pd.Series(map_date_fin).reindex(dc_tmp["id_client"]).values

    hazard_obs = calculer_hazard_observe(fact_tmp, dc_tmp, dim_date)
    delta = logit(CIBLE_CHURN_MENSUEL) - logit(hazard_obs)
    beta0 = beta0 + delta

    print(f"[CALIBRAGE] it={it+1} | hazard_obs={hazard_obs:.4f} | delta_beta0={delta:.3f} -> beta0={beta0:.3f}")

    if abs(hazard_obs - CIBLE_CHURN_MENSUEL) < 0.003:
        break

# Génération finale avec beta0 calibré
fact_abonnement_mensuel, map_date_fin = generer_fact_et_dates_fin(
    dim_client=dim_client,
    ids_clients=ids_clients,
    segs=segs,
    dim_offre=dim_offre,
    seed=SEED_GENERATION + 99,
    beta0=beta0
)

# Date de fin abonnement + statut
dim_client["date_fin_abonnement"] = pd.Series(map_date_fin).reindex(dim_client["id_client"]).values
dim_client["date_fin_abonnement"] = pd.to_datetime(dim_client["date_fin_abonnement"], errors="coerce").dt.date
dim_client["statut_abonnement"] = np.where(dim_client["date_fin_abonnement"].isna(), "Actif", "Résilié")

# Injection NA / outliers
fact_abonnement_mensuel = injecter_bruit_qualite(fact_abonnement_mensuel, seed=SEED_GENERATION + 999)

# Exports CSV
dim_client.to_csv("dim_client.csv", index=False)
dim_offre.to_csv("dim_offre.csv", index=False)
dim_date.to_csv("dim_date.csv", index=False)
fact_abonnement_mensuel.to_csv("fact_abonnement_mensuel.csv", index=False)

print("[OK] Fichiers générés : dim_client.csv, dim_offre.csv, dim_date.csv, fact_abonnement_mensuel.csv")
print("Volumes :", len(dim_client), len(dim_offre), len(dim_date), len(fact_abonnement_mensuel))



## 6) Contrôles de cohérence (QA BI)
Cette section vérifie que :
- aucune observation n’existe après `date_fin_abonnement`
- le nombre de clients actifs par mois est cohérent
- les niveaux de NA et outliers correspondent aux paramètres
- quelques KPI “sanity check” sont plausibles


In [ ]:

# Chargement local (depuis les variables)
dc = dim_client.copy()
dd = dim_date.copy()
do = dim_offre.copy()
fa = fact_abonnement_mensuel.copy()

dc["date_debut_abonnement"] = pd.to_datetime(dc["date_debut_abonnement"], errors="coerce")
dc["date_fin_abonnement"] = pd.to_datetime(dc["date_fin_abonnement"], errors="coerce")
dd["date_mois"] = pd.to_datetime(dd["date_mois"], errors="coerce")

fa = fa.merge(dd[["id_date", "date_mois"]], on="id_date", how="left").merge(
    dc[["id_client", "date_debut_abonnement", "date_fin_abonnement"]],
    on="id_client", how="left"
)

# 1) Observations après churn
viol_after = fa[fa["date_fin_abonnement"].notna() & (fa["date_mois"] > fa["date_fin_abonnement"])]
print("Violations (lignes après date_fin_abonnement):", len(viol_after))

# 2) Actifs par mois (attendu vs fact)
def actifs_attendus(month):
    start_ok = dc["date_debut_abonnement"] <= month
    end_ok = dc["date_fin_abonnement"].isna() | (dc["date_fin_abonnement"] >= month)
    return int((start_ok & end_ok).sum())

attendus = dd.sort_values("date_mois")["date_mois"].apply(actifs_attendus)
observes = fa.groupby("id_date")["id_client"].nunique().reindex(dd["id_date"].sort_values()).values

print("Actifs (attendu) min/max:", int(attendus.min()), int(attendus.max()))
print("Actifs (observé) min/max:", int(np.min(observes)), int(np.max(observes)))
print("Différences (doit être 0):", int(np.max(np.abs(observes - attendus.values))))

# 3) Churn global sur fenêtre
churn_rate = dc["date_fin_abonnement"].notna().mean()
print("Taux churn_client sur la fenêtre:", round(float(churn_rate), 3))

# 4) NA
taux_na = fa[["mrr_eur", "heures_utilisation", "score_satisfaction"]].isna().mean()
print("Taux NA (mrr/usage/satisfaction):")
print(taux_na.round(4))

# 5) Outliers (max)
max_vals = fa[["mrr_eur", "heures_utilisation", "nb_tickets_support"]].max(numeric_only=True)
print("Max (mrr / usage / tickets):")
print(max_vals)

# 6) KPI par offre (sanity)
fa2 = fa.merge(do[["id_offre", "nom_offre", "prix_mensuel_base_eur", "prix_par_licence_eur"]], on="id_offre", how="left")
kpi_offre = (fa2.groupby("nom_offre")
             .agg(clients=("id_client", "nunique"),
                  mrr_moy=("mrr_eur", "mean"),
                  licences_moy=("nb_licences", "mean"),
                  usage_moy=("heures_utilisation", "mean"),
                  tickets_moy=("nb_tickets_support", "mean"),
                  satisfaction_moy=("score_satisfaction", "mean"))
             .sort_values("clients", ascending=False))
print("\nKPI par offre (aperçu) :")
display(kpi_offre)

# Corrélation attendue (satisfaction vs tickets) sur l’ensemble
corr = fa2[["score_satisfaction", "nb_tickets_support", "nb_escalades", "heures_utilisation"]].corr(numeric_only=True)
print("\nCorrélations globales (attendu: sat ↓ quand tickets/escalades ↑ ; sat ↑ quand usage ↑) :")
display(corr.loc["score_satisfaction"])
